# Pressure-Only Context Features

The broad pocket-context experiment added QB movement along with pressure indicators. This follow-up isolates six pressure features: nearest rusher distance at snap, minimum and mean distance before release, closing rate, and the share of pre-release frames within three and five yards. The target remains snap-to-release `delta_sep`.

We compare static ridge, dynamic defender-context ridge, pressure-only ridge, and the broad pocket ridge. The same five game-grouped folds and week-balanced receiver reliability halves are used for each model.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from separation_over_expected.feature_schema import PRESSURE_CONTEXT_FEATURES
from separation_over_expected.reports import read_csv_rows

EXPERIMENT_DIR = PROJECT_ROOT / "data" / "processed" / "dynamic_features" / "pressure_context" / "cross_validation"
METRICS = read_csv_rows(EXPERIMENT_DIR / "oof_metrics_wr.csv")
RELIABILITY = read_csv_rows(EXPERIMENT_DIR / "receiver_oof_reliability_wr.csv")
FEATURES = PRESSURE_CONTEXT_FEATURES
FEATURES

('qb_nearest_rusher_dist_snap', 'qb_nearest_rusher_min_dist_pre_release', 'qb_nearest_rusher_mean_dist_pre_release', 'qb_rusher_closing_rate_pre_release', 'qb_rusher_within_3yd_frame_share_pre_release', 'qb_rusher_within_5yd_frame_share_pre_release')

## Out-of-fold route prediction

The OOF summary aggregates the same 20,415 routes scored by models trained on other games. Compare pressure-only with dynamic context to see whether pressure adds route-level signal, then compare broad pocket context with pressure-only to see whether QB movement adds anything beyond those pressure features.

In [4]:
OOF = [row for row in METRICS if row["fold"] == "OOF_ALL"]
OOF

[{'fold': 'OOF_ALL', 'model': 'ridge_context', 'mae': '1.390', 'rmse': '1.897', 'r2': '0.493', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.341', 'n': '20415'}, {'fold': 'OOF_ALL', 'model': 'ridge_dynamic_context', 'mae': '1.363', 'rmse': '1.869', 'r2': '0.508', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.342', 'n': '20415'}, {'fold': 'OOF_ALL', 'model': 'ridge_pressure_context', 'mae': '1.362', 'rmse': '1.867', 'r2': '0.509', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.341', 'n': '20415'}, {'fold': 'OOF_ALL', 'model': 'ridge_pocket_context', 'mae': '1.362', 'rmse': '1.867', 'r2': '0.509', 'bias': '0.001', 'mean_actual': '-2.341', 'mean_predicted': '-2.341', 'n': '20415'}]

In [5]:
FOLD_RESULTS = [row for row in METRICS if row["fold"] != "OOF_ALL"]
FOLD_RESULTS

[{'fold': '1', 'model': 'ridge_context', 'mae': '1.363', 'rmse': '1.853', 'r2': '0.496', 'bias': '0.011', 'mean_actual': '-2.276', 'mean_predicted': '-2.287', 'n': '4972'}, {'fold': '1', 'model': 'ridge_dynamic_context', 'mae': '1.335', 'rmse': '1.826', 'r2': '0.511', 'bias': '0.011', 'mean_actual': '-2.276', 'mean_predicted': '-2.287', 'n': '4972'}, {'fold': '1', 'model': 'ridge_pocket_context', 'mae': '1.334', 'rmse': '1.823', 'r2': '0.513', 'bias': '0.007', 'mean_actual': '-2.276', 'mean_predicted': '-2.283', 'n': '4972'}, {'fold': '1', 'model': 'ridge_pressure_context', 'mae': '1.334', 'rmse': '1.824', 'r2': '0.512', 'bias': '0.006', 'mean_actual': '-2.276', 'mean_predicted': '-2.282', 'n': '4972'}, {'fold': '2', 'model': 'ridge_context', 'mae': '1.409', 'rmse': '1.906', 'r2': '0.503', 'bias': '-0.096', 'mean_actual': '-2.583', 'mean_predicted': '-2.487', 'n': '3942'}, {'fold': '2', 'model': 'ridge_dynamic_context', 'mae': '1.377', 'rmse': '1.875', 'r2': '0.519', 'bias': '-0.084', 

## Receiver-level reliability

Each receiver needs at least 20 routes in both game halves. We report Pearson agreement in residual levels and Spearman agreement in rankings. The paired bootstrap compares the models on the same receivers.

In [7]:
RELIABILITY

[{'comparison': 'ridge_context', 'eligible_receivers': '115', 'pearson': '0.430880', 'spearman': '0.220556', 'pearson_ci_lower': '0.071785', 'pearson_ci_upper': '0.672905', 'spearman_ci_lower': '0.032200', 'spearman_ci_upper': '0.386949', 'minimum_routes_each_half': '20'}, {'comparison': 'ridge_dynamic_context', 'eligible_receivers': '115', 'pearson': '0.451857', 'spearman': '0.209469', 'pearson_ci_lower': '0.079452', 'pearson_ci_upper': '0.691667', 'spearman_ci_lower': '0.022494', 'spearman_ci_upper': '0.384713', 'minimum_routes_each_half': '20'}, {'comparison': 'ridge_pressure_context', 'eligible_receivers': '115', 'pearson': '0.434050', 'spearman': '0.204119', 'pearson_ci_lower': '0.065690', 'pearson_ci_upper': '0.676201', 'spearman_ci_lower': '0.014663', 'spearman_ci_upper': '0.377914', 'minimum_routes_each_half': '20'}, {'comparison': 'ridge_pocket_context', 'eligible_receivers': '115', 'pearson': '0.423567', 'spearman': '0.184195', 'pearson_ci_lower': '0.046663', 'pearson_ci_uppe

## Interpretation

Pressure-only produces a small route-level gain over dynamic context (OOF RMSE 1.869 to 1.867; R² 0.508 to 0.509). Its receiver Pearson reliability is 0.434 versus 0.452 for dynamic context; the paired difference is -0.018 (95% interval -0.035 to 0.005), so this evaluation neither shows a reliability gain nor clear evidence of a loss. Spearman reliability is also slightly lower, with a paired interval spanning zero.

The broad pocket model has effectively the same route RMSE as pressure-only, but lower receiver reliability (Pearson 0.424; Spearman 0.184). Relative to pressure-only, the broad feature set's paired reliability differences are -0.010 Pearson (95% interval -0.024 to -0.003) and -0.020 Spearman (95% interval -0.040 to -0.002). The QB movement additions therefore do not help this receiver-stability check.

Takeaway: pressure-only is a more promising and narrower predictor than the broad pocket set, but it does not beat the existing dynamic model on receiver reliability. Keep dynamic context as the current preferred model. The modest route-prediction gain alone is not enough to claim better player evaluation. Results cover only the first eight weeks of one season.